In [0]:
silver_dividend = spark.sql("SELECT * from plstocks.silver_dividend_history")
sector_lookup = spark.sql("SELECT * FROM plstocks.silver_sector_lookup")
silver_financial_reports = spark.sql("SELECT * FROM plstocks.silver_financial_reports")

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import IntegerType
import pandas as pd
import datetime

current_year = datetime.datetime.now().year

dividends_with_sectors = silver_dividend.join(sector_lookup, on='ticker')
dividends_with_sectors = dividends_with_sectors.withColumnRenamed("dividend_year", "year")

dividends_financials = dividends_with_sectors.join(
    silver_financial_reports.select('ticker', 'year', 'net_profit'),
    on=['year', 'ticker'],
    how='left'
).withColumn(
    'payout_ratio',
    F.col('dividend_value') / F.col('net_profit') * 100
)

w = Window.partitionBy('ticker').orderBy('year')

dividends_growth = dividends_financials.withColumn(
    'prev_dividend',
    F.lag('dividend_per_share').over(w)
).withColumn(
    'dividend_growth',
    (F.col('dividend_per_share') - F.col('prev_dividend')) / F.col('prev_dividend')
).withColumn(
    'paid_dividend',
    F.when(F.col('dividend_per_share') > 0, 1).otherwise(0)
).withColumn(
    'dividend_increased',
    F.when(F.col('dividend_growth') > 0, 1).otherwise(0)
)

years_paid = dividends_growth.filter("paid_dividend = 1").groupBy("ticker", "sector").agg(
    F.collect_list("year").alias("paid_years")
)

years_growth = dividends_growth.filter("dividend_increased = 1").groupBy("ticker").agg(
    F.collect_list("year").alias("growth_years")
)

def longest_consecutive_ending_streak(years):
    if years is None or len(years) == 0:
        return 0
    years = sorted(set(years))
    max_streak = 0
    current_streak = 1
    for i in range(1, len(years)):
        if years[i] == years[i - 1] + 1:
            current_streak += 1
        else:
            current_streak = 1
        if years[i] in [current_year, current_year - 1, current_year - 2]:
            max_streak = max(max_streak, current_streak)
    if years[-1] not in [current_year, current_year - 1, current_year - 2]:
        return 0
    return max_streak

@F.pandas_udf(IntegerType())
def calc_consec_streak(paid_years: pd.Series) -> pd.Series:
    return paid_years.apply(longest_consecutive_ending_streak)

@F.pandas_udf(IntegerType())
def calc_growth_streak(growth_years: pd.Series) -> pd.Series:
    return growth_years.apply(longest_consecutive_ending_streak)

years_paid = years_paid.withColumn(
    "consecutive_years_paying", calc_consec_streak("paid_years")
)

years_growth = years_growth.withColumn(
    "consecutive_years_growing", calc_growth_streak("growth_years")
)

last_3yr = dividends_growth.withColumn(
    "latest_year", F.lit(current_year)
).filter(
    F.col("year") >= F.col("latest_year") - 2
)

growth_3yr = last_3yr.groupBy("ticker").agg(
    F.avg("dividend_growth").alias("avg_3yr_dividend_growth")
)

latest_payout = dividends_growth.groupBy("ticker").agg(
    F.max("payout_ratio").alias("latest_payout_ratio")
)

result = years_paid.join(years_growth, on="ticker", how="left").join(
    growth_3yr, on="ticker", how="left"
).join(
    latest_payout, on="ticker", how="left"
).select(
    "ticker",
    "sector",
    F.col("paid_years").alias("years"),
    "consecutive_years_paying",
    "consecutive_years_growing",
    "avg_3yr_dividend_growth",
    "latest_payout_ratio"
)


In [0]:
result.write.mode("overwrite").saveAsTable("plstocks.gold_dividend_yield_analysis")